In [24]:
from pydantic import BaseModel, Field, validator, ValidationError
from typing import Optional, List
from enum import Enum
from openai import OpenAI
from dotenv import load_dotenv
import json
import os

load_dotenv()
client = OpenAI()

In [25]:
# === SCHRITT 2: MODELLE ===
class ProductCategory(str, Enum):
    ELECTRONICS = "electronics"
    CLOTHING = "clothing"
    FOOD = "food"
    HOME = "home"
    OTHER = "other"

class ProductRequest(BaseModel):
    name: str = Field(..., min_length=2, max_length=100)
    price: float = Field(..., gt=0)
    category: ProductCategory
    description: Optional[str] = Field(None, max_length=500)
    additional_info: Optional[dict] = None
    target_audience: Optional[str] = Field(None, max_length=100)

    @validator('name')
    def name_not_empty(cls, v):
        if not v.strip():
            raise ValueError('Name darf nicht nur Leerzeichen enthalten')
        return v.strip()

    @validator('description')
    def description_meaningful(cls, v):
        if v is not None and len(v.strip()) < 10:
            raise ValueError('Beschreibung muss mindestens 10 Zeichen haben')
        return v

class ProductListing(BaseModel):
    title: str
    short_description: str
    key_features: List[str]
    call_to_action: str

/var/folders/2g/5q_lj5g51f19wyt7z8k7qxs00000gn/T/ipykernel_7229/2149555152.py:17: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  @validator('name')
/var/folders/2g/5q_lj5g51f19wyt7z8k7qxs00000gn/T/ipykernel_7229/2149555152.py:23: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  @validator('description')


In [26]:
# === SCHRITT 3: VALIDIERUNG ===
def load_and_validate(data: dict):
    try:
        product = ProductRequest(**data)
        return product, []
    except ValidationError as e:
        errors = [f"  Feld '{err['loc'][0]}': {err['msg']}" for err in e.errors()]
        return None, errors

In [27]:
# === SCHRITT 4: API + PIPELINE ===
def generate_listing(product: ProductRequest):
    prompt = f"""
    Erstelle eine Produktbeschreibung für einen Online-Shop.
    
    Produkt: {product.name}
    Preis: ${product.price}
    Kategorie: {product.category}
    Beschreibung: {product.description or 'keine Angabe'}
    Zielgruppe: {product.target_audience or 'allgemein'}
    Zusatzinfos: {product.additional_info or {}}
    
    Antworte NUR als JSON (kein Markdown, kein Text darum):
    {{
        "title": "Produkttitel (max 60 Zeichen)",
        "short_description": "Kurzbeschreibung (max 150 Zeichen)",
        "key_features": ["Feature 1", "Feature 2", "Feature 3"],
        "call_to_action": "Handlungsaufforderung"
    }}
    """
    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "Du bist ein E-Commerce-Texter. Antworte immer nur mit validem JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )
        raw_output = response.choices[0].message.content
        output_data = json.loads(raw_output)
        listing = ProductListing(**output_data)
        return listing
    except Exception as e:
        print(f"  ✗ Fehler: {e}")
        return None


def process_request(data: dict):
    print("-" * 40)
    product, errors = load_and_validate(data)

    if not product:
        print("✗ Validierung fehlgeschlagen:")
        for err in errors:
            print(err)
        print("→ API-Call abgebrochen")
        return

    print(f"✓ Input valide: {product.name}")
    print("→ Sende an ChatGPT...")
    listing = generate_listing(product)

    if not listing:
        print("✗ Kein gültiges Listing erhalten")
        return

    print(f"\n✓ Listing generiert:")
    print(f"  Titel:        {listing.title}")
    print(f"  Beschreibung: {listing.short_description}")
    print(f"  Features:")
    for f in listing.key_features:
        print(f"    • {f}")
    print(f"  CTA:          {listing.call_to_action}")

In [28]:
# === TEST-DATEN ===
valid_product = {
    "name": "Premium Kaffeemaschine",
    "price": 129.99,
    "category": "home",
    "description": "Vollautomatische Espressomaschine für zuhause",
    "target_audience": "Kaffeeliebhaber",
    "additional_info": {"watt": 1500, "tank_liter": 1.8}
}

invalid_product = {
    "name": "X",
    "price": -20,
    "category": "gadgets"
}


In [29]:
# === AUSFÜHREN ===
print("TEST 1: Valides Produkt")
process_request(valid_product)

print("\nTEST 2: Invalides Produkt")
process_request(invalid_product)

TEST 1: Valides Produkt
----------------------------------------
✓ Input valide: Premium Kaffeemaschine
→ Sende an ChatGPT...

✓ Listing generiert:
  Titel:        Premium Kaffeemaschine
  Beschreibung: Vollautomatische Espressomaschine für zuhause - ideal für Kaffeeliebhaber
  Features:
    • 1500 Watt Leistung
    • 1.8 Liter Wassertank
    • Hochwertige Verarbeitung
  CTA:          Jetzt kaufen und den perfekten Kaffee genießen!

TEST 2: Invalides Produkt
----------------------------------------
✗ Validierung fehlgeschlagen:
  Feld 'name': String should have at least 2 characters
  Feld 'price': Input should be greater than 0
  Feld 'category': Input should be 'electronics', 'clothing', 'food', 'home' or 'other'
→ API-Call abgebrochen


In [30]:
# valid_product.json erstellen
import json

with open("valid_product.json", "w") as f:
    json.dump(valid_product, f, indent=4)

with open("invalid_product.json", "w") as f:
    json.dump(invalid_product, f, indent=4)

In [31]:
def load_and_validate_from_file(filepath: str):
    with open(filepath, 'r') as f:
        data = json.load(f)
    return load_and_validate(data)  # bestehende Funktion nutzen